<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex08.1-stationary-heat/Ex08.1_00_geometry_check.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_08.1 · Notebook 00 — Geometry and Setup

**Paired with L8.1 · Stationary Heat Transfer**

**Read and run; you are not asked to rewrite this.**

Stationary conduction on a plate with a cooling hole:

$$\frac{\partial^2 T}{\partial x^2} + \frac{\partial^2 T}{\partial y^2}
+ \frac{Q}{k} = 0$$

`course_core.py` and `pinn_core.py` are the same two modules you used in
Ex_07.1 and Ex_07.2. `problem.py` adds only geometry: sampling around a hole,
the level-set multiplier, outward normals on a curved boundary, and a flux
balance.

Run it top to bottom. Nothing here is yours to write. Its job is to establish,
before you spend an hour on anything else, that the tools import, that the
samplers put points where they claim, and that the multiplier and the normals
are the shapes and the signs they are supposed to be.

If a cell fails here, fix it before going on.

---

## 0 · Versions

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex08.1-stationary-heat/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
import sys
import numpy as np
import matplotlib
import torch

print("python     ", sys.version.split()[0])
print("numpy      ", np.__version__)
print("matplotlib ", matplotlib.__version__)
print("torch      ", torch.__version__)
print("cuda        available:", torch.cuda.is_available(), " (not needed)")

**What you should see.** Four version numbers. Any Python from 3.9 and any
PyTorch from 2.0 will do, and **no GPU is required anywhere in Part 2**.

---

## 1 · The three modules

Every Part 2 exercise has the same three files beside it. The first two are
identical in every set; only the third changes.

| | |
|---|---|
| `course_core.py` | shared by the whole course — `set_seed`, `MLP`, `to_tensor`, `check` |
| `pinn_core.py` | the PDE machinery — `grad`, `d2`, samplers, `train_two_stage` |
| `problem.py` | **this** problem — the hole, the multiplier, the normals, the flux balance |

In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

**What you should see.** `device: cpu` on most machines and
`dtype: torch.float64`.

Double precision is deliberate. A second derivative of a network is a
difference of differences; in float32 the residual can be dominated by rounding
long before it is dominated by the model.

---

## 2 · The plate and the hole

Nothing below needs torch — the geometry is arithmetic, and `problem.py` keeps
it that way on purpose so it can be checked without a training run.

In [ ]:
pb.describe_problem()

**What you should see.**

```
  plate           : 1.000 x 1.000   (unit square)
  hole            : ellipse at (0.50, 0.50),  a = 0.18,  b = 0.11
  hole area       : 0.06220   = 6.22% of the plate
  material area   : 0.93780
  hole perimeter  : 0.92438   (Ramanujan)
  aspect ratio a/b: 1.636   -> the ends are 4.38x as curved as the sides
  generated heat  : Q/k x 0.93780   must leave through 0.92438 of wall
```

The last line is the flux balance stated in advance, and it is the check
notebook 03 will make. Everything generated in an area of 0.938 has to leave
through 0.924 of hole wall, because in steady state there is nowhere else for
it to go — the outer edges are insulated.

The problem is posed on the **unit square** already. That is not a
simplification for its own sake: slide 8 makes the point that the multiplier
$\xi(1-\xi)\eta(1-\eta)$ only vanishes on the boundary if the domain is the
unit square, so scaling comes before any trial solution is written.

---

## 3 · Sampling a plate with a hole

Three point sets, three different jobs. The interior points carry the PDE, the
hole points carry the fixed temperature, and the outer edge points carry the
zero-flux condition.

The samplers return **NumPy arrays**, not tensors — which is why they can be
plotted directly, with no `.detach().cpu().numpy()` anywhere. Call
`to_tensor(...)` at the point of use instead, and `requires_grad=True` wherever
the network gets differentiated.

In [ ]:
xy_f = pb.sample_plate_with_hole(1200)
xy_h = pb.sample_ellipse_boundary(80)
xy_o = pb.sample_outer_edges(30)
for nm, s in [("interior", xy_f), ("hole", xy_h), ("outer", xy_o)]:
    print(f"{nm:>9s}: {s.shape[0]:5d} points")

check_shape("interior", xy_f, (1200, 2))
check_shape("hole", xy_h, (80, 2))
check_shape("outer", xy_o, (120, 2))

plt.figure(figsize=(5, 5))
plt.scatter(xy_f[:, 0], xy_f[:, 1], s=4, label="interior")
plt.scatter(xy_o[:, 0], xy_o[:, 1], s=12, label="outer edges")
plt.scatter(xy_h[:, 0], xy_h[:, 1], s=14, label="hole")
plt.gca().set_aspect("equal"); plt.legend(); plt.show()

**What you should see.** `1200`, `80` and `120` points, three `PASS`
lines, and a filled unit square with a clean elliptical void in the middle,
ringed by hole points and bordered by edge points.

`sample_outer_edges(30)` gives 120 points, not 30: `boundary_points` places
`n_per_edge` on **each** of the four edges.

The void has a small margin around it. `sample_plate_with_hole` rejects against
an ellipse inflated by 5%, so no collocation point lands where the multiplier —
and therefore the whole trial solution — is zero. A residual evaluated there
would be identically satisfied and would teach the network nothing.

---

## 4 · Arc length beats angle

Uniform spacing in the parameter angle bunches points at the flat sides of an
ellipse and starves the high-curvature ends — exactly where the flux
concentrates. Compare the spacing.

In [ ]:
P = pb.sample_ellipse_boundary(60)
t = np.linspace(0, 2 * np.pi, 60, endpoint=False)
A = np.column_stack([pb.HOLE["xc"] + pb.HOLE["a"] * np.cos(t),
                     pb.HOLE["yc"] + pb.HOLE["b"] * np.sin(t)])
for nm, Q in [("arc length", P), ("angle", A)]:
    d = np.hypot(*np.diff(np.vstack([Q, Q[:1]]), axis=0).T)
    print(f"{nm:>11s}: spacing spread = {d.std() / d.mean():.4f}")

**What you should see.**

```
 arc length: spacing spread = 0.0006
      angle: spacing spread = 0.1673
```

Two orders of magnitude. The angular sample's gaps are all at the ends of the
major axis, and the ends of the major axis are where this ellipse is
`4.38x` as curved as its sides — which is where the flux crowds together and
where a flux condition most needs points.

---

## 5 · The multiplier and the normals

`hole_multiplier` is the level-set function: zero on the hole, positive in the
material. Multiplying the network by it hard-enforces a fixed hole temperature
with no pre-training — the analytic shortcut of slide 15.

It is plain arithmetic, so it works on the NumPy points directly.
`ellipse_normal` is not: it needs `torch.sqrt`, so the points go through
`to_tensor` first.

In [ ]:
print("multiplier on the hole (must be ~0):",
      f"{np.abs(pb.hole_multiplier(xy_h)).max():.3e}")
print("multiplier in the material (must be > 0):",
      f"{pb.hole_multiplier(xy_f).min():.4f}")

nx, ny = pb.ellipse_normal(to_tensor(xy_h))
plt.figure(figsize=(5, 5))
plt.quiver(xy_h[:, 0], xy_h[:, 1],
           to_numpy(nx).ravel(), to_numpy(ny).ravel(),
           scale=18, color="tab:orange")
plt.gca().set_aspect("equal"); plt.title("normals point OUT of the material")
plt.show()

**What you should see.** `8.882e-16` on the hole — machine zero, not
"small" — and `0.1113` in the material, and a ring of arrows pointing
**inwards, into the void**.

If the arrows point into the material, the sign is wrong — see slide 16. The
material's outward normal is the *inward* normal of the hole, and the minus
sign in `ellipse_normal` is what puts it there. A sign error here produces a
field that looks entirely plausible and is inverted.

---

## 6 · Ready

You have checked the tools, the geometry, the samplers, the multiplier and the
normals. Nothing later in Ex_08.1 depends on anything you have not just seen.

Next: **notebook 01**, where the formulation is verified on a manufactured
solution before any of this geometry is switched on.